# Run: generate the simulated dataset

Replays games from `configs/data_generator.yaml` in randomized sessions and writes
`gambit_vision/data/crops/sim_*/{squares,frames,manifest.csv}` (contracts.md format), for the
ResNet-18 sim-only and mixed training runs.

**TODO**
- [ ] finish `grab_frame`, `extract_square_crops`, `apply_lighting`, `apply_camera_pose` in data_generator
- [ ] add a games PGN at `games.source`
- [ ] run 1 session x 1 game first, inspect crops, then the full set
- [ ] write split assignment (train/val/test by session) where gambit_vision's dataset expects it

In [ ]:
%run ./data_generator.ipynb

In [ ]:
cfg = DATA_CFG
games_path = REPO / cfg["games"]["source"]
games = []
for i, g in enumerate(read_pgn_games(games_path)):
    if i >= cfg["games"]["max_games"]:
        break
    games.append((f"g{i:04d}", str(g)))
print(len(games), "games")

In [ ]:
backend = GzCliBackend()
n = cfg["sessions"]["games_per_session"]
for s in range(cfg["sessions"]["count"]):
    out = generate_session(s, games[s * n:(s + 1) * n], backend, seed=cfg["games"]["seed"])
    print("wrote", out)

## Inspect
Per-class counts per session (empty dominates; check the piece classes are not too rare).

In [ ]:
import pandas as pd
root = REPO / cfg["output"]["root"]
df = pd.concat([pd.read_csv(p) for p in sorted(root.glob("sim_*/manifest.csv"))])
print(df.groupby(["session_id", "label"]).size().unstack(fill_value=0))